# 초구 예측 — model_04

- 1단계 항목: 위치 기준점 **공 수 가중 → 투수 동일 가중**
  - 기준점 = 분석 대상 투수 평균 (구종×타자 손, 전날까지 반감기 가중) -> 지금은 공 하나가 한 표라 많이 던진 투수 쪽으로 쏠림
  - 이유: 한 투수의 위치 습관이 기준점을 끌고 가면, 다른 투수를 예측할 때 그 투수 쪽으로 당기게 됨
- 바꾸는 것: 기준점 안 투수별 무게 + 위치 파라미터 다시 고름
  - 구종은 model_01, 24칸 적분은 model_03(공분산 반영) 그대로
- 비교 대상: model_03 (2019로 고르고, 2020은 compare 2-1 부트스트랩으로 확인)
- 이름: 이 노트북부터 "리그 평균" 대신 "분석 대상 투수 평균" (데이터가 투수 41명 공뿐)

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys
if not os.path.exists("fp_common.py"):  # model/ 폴더에서 열면 한 단계 위(데이터·fp_common·results 있는 곳)로
    os.chdir("..")
sys.path.insert(0, os.getcwd())  # fp_common을 찾도록
import numpy as np
import pandas as pd
import fp_common as fc

MODEL = "model_04"

m, names = fc.load_data()
fp, y_type, y_zone, val, test = fc.first_pitches(m)
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]
print(f"초구 {len(fp):,}개 (검증 {val.sum():,} / 평가 {test.sum():,})")

초구 50,709개 (검증 13,458 / 평가 12,615)


## 1. 파라미터

- 구종: model_01 값 고정
- 위치: model_03 값에서 시작 → 아래 3-2에서 기준점 무게와 함께 다시 고르고 1등으로 덮어씀

In [2]:
# ① 구종 (model_01, 고정): 반감기(공 수), 분석 대상 투수 평균 쪽 당김, 타자 손 쪽 당김
TYPE_HL, TYPE_ALPHA, TYPE_HAND_K = 25.0, 5.48, 5.48
# ② 위치 기준점 (model_03에서 시작, 다시 고름): 전날까지 구종×손별 반감기(그 그룹의 공 수) 가중
LOC_REF, LOC_REF_HL = "decay", 173.0
# ③ 기준점 안 투수 무게: 투수 무게 = n / (n + REF_C), n = 그 투수의 (반감기 가중) 공 수
#    REF_C = 0 투수 동일 가중 / 클수록 공 수 쪽 / None = 공 수 가중 (model_03과 같음)
REF_C = None
# ② 위치 (model_03에서 시작, 다시 고름): 반감기(공 수), 투수×구종 / 투수×구종×타자 손 당김
LOC_HL, K_TYPE, K_HAND = 35.4, 1000.0, 17.3
# ③ 24칸 적분: 좌우 구간마다 수치 적분 점 개수 (가우스-르장드르, fc.zone_from_gauss_cov 기본값)
ZONE_M = 32

## 2. ① 구종 (model_01과 같음)

In [3]:
def league_type(year):
    return fp.loc[fp["GYEAR"] < year, CLASSES].mean().to_numpy()

def type_online(q, year, half_life, alpha, hand_k):
    S, N = fc.lookup(fc.prior_table(fp, ["pitcher_code"], CLASSES, half_life), q, ["pitcher_code"], CLASSES)
    P = fc.shrink(S, N[:, :1], league_type(year), alpha)
    if hand_k:
        keys = ["pitcher_code", "same_side"]
        S2, N2 = fc.lookup(fc.prior_table(fp, keys, CLASSES, half_life), q, keys, CLASSES)
        P = fc.shrink(S2, N2[:, :1], P, hand_k)
    return P

## 3. ② 위치

- 당기는 순서·24칸 적분은 model_03과 같음 (기준점 → 투수×구종 → 투수×구종×타자 손, 공분산 반영)
- 바뀐 것: 기준점 안 투수별 무게
  - model_03: 구종×손 그룹의 공을 전날까지 반감기 가중 평균 = 투수마다 (그 투수 가중 평균) × (가중 공 수 n) 로 합친 것
  - model_04: 투수 무게를 n 대신 n / (n + c) 로
    - c = 0: 공이 조금이라도 있으면 모든 투수 같은 무게 (투수 동일 가중)
    - c 클수록 n에 비례 -> c = None(∞)이면 model_03과 정확히 같음 (대조군)
    - c = 투수 한 명이 최대 무게의 절반을 받는 (가중) 공 수
  - 반감기는 model_03처럼 그 구종×손 그룹의 공 수로 셈 -> 투수가 쉬는 동안 그 투수 무게도 같이 줄어듦 (최근 분석 대상 투수 경향 유지)

In [5]:
# 24칸 적분 함수는 fp_common으로 옮김: fc.zone_from_gauss_cov(g, q, m=32) (계산 설명은 위, 코드 주석은 fp_common.py)
def zone_probs(g, q, cov=True):  # 24칸 확률: cov=True 공분산 반영 (model_03) / False 독립 가정 (model_02까지)
    return fc.zone_from_gauss_cov(g, q, ZONE_M) if cov else fc.zone_from_gauss(g, q)

In [6]:
L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개(x, 높이 비율, 제곱, 곱) + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]  # 투수×구종 / 투수×구종×손
KR = ["pitch_type", "same_side"]  # 기준점 그룹: 분석 대상 투수 구종×타자 손
_REF = {}  # 기준점 계산 캐시 (반감기·무게·시즌이 같으면 다시 만들지 않음)

def peq_table(ref_hl):  # 투수×구종×손×날짜별 누적 합계 (공마다 2^(그룹 안 순번/반감기) 가중), 그룹×날짜별 그 날까지 그룹 공 수
    if ref_hl in _REF:
        return _REF[ref_hl]
    d = fp.sort_values(["game_date", "gameID", "pit_no"]).copy()  # 시간 순서
    L = d.groupby(KR, observed=True).cumcount().to_numpy()  # 구종×손 그룹 안 순번 (0, 1, 2, …)
    a = 2.0 ** (L / ref_hl)  # 공마다 가중치: 나중 공일수록 큼 (전날 시점에서 2^(−그 시점 순번/반감기)를 곱하면 반감기 가중치)
    X = d[L_COLS].to_numpy(float)
    ok = ~np.isnan(X)  # 값 있는 칸
    X = np.where(ok, X, 0.0)
    cols = [f"S_{c}" for c in L_COLS] + [f"N_{c}" for c in L_COLS]
    t = pd.DataFrame(np.c_[a[:, None] * X, a[:, None] * ok], columns=cols)  # 가중 합계, 가중 개수
    for k in K2 + ["game_date"]:
        t[k] = d[k].to_numpy()
    t = t.groupby(K2 + ["game_date"], observed=True)[cols].sum().groupby(level=K2, observed=True).cumsum().reset_index()  # 투수×구종×손별 그 날까지 누적
    gl = d.groupby(KR + ["game_date"], observed=True).size().groupby(level=KR, observed=True).cumsum().reset_index(name="L_end")  # 그룹별 그 날까지 공 수
    for tab, keys in [(t, K2), (gl, KR)]:
        for k in keys:
            tab[k] = tab[k].astype(fp[k].dtype)
    _REF[ref_hl] = (t.sort_values("game_date"), gl.sort_values("game_date"))
    return _REF[ref_hl]

def ref_pitcher_weight(year, ref_hl, c):  # 기준점 표 (구종×손×날짜, 29): 투수별 반감기 가중 평균을 무게 n_p/(n_p + c)로 합침, 그 시즌 경기 날짜 전부
    key = (year, ref_hl, c)
    if key in _REF:
        return _REF[key]
    t, gl = peq_table(ref_hl)
    dates = np.sort(fp.loc[fp["GYEAR"] == year, "game_date"].unique())  # 그 시즌 경기 날짜
    grid = t[K2].drop_duplicates().merge(pd.DataFrame({"game_date": dates}), how="cross")  # (투수×구종×손) × 날짜
    r = pd.merge_asof(grid.sort_values("game_date"), t, on="game_date", by=K2, allow_exact_matches=False)  # 전날까지 누적 (같은 날 제외)
    r = pd.merge_asof(r.sort_values("game_date"), gl, on="game_date", by=KR, allow_exact_matches=False)  # 전날까지 그룹 공 수
    S = r[[f"S_{c_}" for c_ in L_COLS]].fillna(0).to_numpy()
    N = r[[f"N_{c_}" for c_ in L_COLS]].fillna(0).to_numpy()
    mean = S / np.maximum(N, 1e-300)  # 투수별 반감기 가중 평균 (가중치 공통 배수는 나눗셈에서 지워짐)
    n_p = N * 2.0 ** (-r["L_end"].fillna(0).to_numpy() / ref_hl)[:, None]  # 투수별 반감기 가중 공 수 (model_03 기준점에서의 몫)
    w = np.where(N > 0, n_p if c is None else n_p / (n_p + c + 1e-300), 0.0)  # 투수 무게 (c=None: 공 수 그대로 = model_03)
    agg = pd.DataFrame(np.c_[w * mean, w])
    for k in ["pitch_type", "same_side", "game_date"]:
        agg[k] = r[k].to_numpy()
    g = agg.groupby(["pitch_type", "same_side", "game_date"], observed=True).sum()  # 구종×손×날짜별 투수 합
    k = len(L_COLS)
    num, den = g.iloc[:, :k].to_numpy(), g.iloc[:, k:].to_numpy()
    _REF[key] = pd.DataFrame(np.where(den > 0, num / np.maximum(den, 1e-300), np.nan), index=g.index)  # 무게 평균
    return _REF[key]

def loc_league(q, year, ref="all", ref_hl=None, ref_c=None):  # 기준점 (n, 29): "all" 지난 시즌 전부 / "last" 직전 시즌만 / "decay" 전날까지 반감기 ref_hl(공 수) 가중, ref_c 투수 무게
    past = fp[fp["GYEAR"] < year]  # 지난 시즌까지
    overall = past[L_COLS].mean().to_numpy()  # 그 구종×손 기록이 없을 때 쓰는 분석 대상 투수 전체 평균 (지난 시즌까지)
    if ref == "decay":
        tab = ref_pitcher_weight(year, ref_hl, ref_c)
        P = tab.reindex(pd.MultiIndex.from_frame(q[["pitch_type", "same_side", "game_date"]])).to_numpy()  # 공마다 그 구종×손×날짜 행
        return np.where(np.isnan(P), overall, P)  # 기록 없으면 분석 대상 투수 전체 평균
    if ref == "last":
        past = fp[fp["GYEAR"] == year - 1]  # 직전 시즌만
    lg = past.groupby(KR)[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[KR]))  # 공마다 그 구종×손 행
    return pr.fillna(pd.Series(overall, index=L_COLS)).to_numpy()

def loc_online(q, year, half_life, k_type, k_hand, ref="all", ref_hl=None, ref_c=None):  # 위치 모멘트 + 24칸 비율 (n, 29): L2
    P = loc_league(q, year, ref, ref_hl, ref_c)  # 출발: 기준점
    S1, N1 = fc.lookup(fc.prior_table(fp, K1, L_COLS, half_life), q, K1, L_COLS)
    P = fc.shrink(S1, N1, P, k_type)  # 투수×구종 쪽으로 k_type만큼
    S2, N2 = fc.lookup(fc.prior_table(fp, K2, L_COLS, half_life), q, K2, L_COLS)
    return fc.shrink(S2, N2, P, k_hand)  # 투수×구종×손 쪽으로 k_hand만큼

def loc_given_type(q, year, t, half_life, k_type, k_hand, ref="all", ref_hl=None, cov=True, ref_c=None):  # 구종이 t라고 가정했을 때 24칸 확률 (n, 24)과 좌표 정규분포
    qt = fc.as_type(q, t)  # 구종만 t로 바꾼 사본
    g = fc.gauss(loc_online(qt, year, half_life, k_type, k_hand, ref, ref_hl, ref_c), qt)  # 모멘트 → 정규분포 (높이는 그 타자 존으로 cm 환산)
    return zone_probs(g, qt, cov), g  # 정규분포를 24칸 위에서 적분 (cov=True 공분산 반영)

### 3-1. 확인: 대조군 재현, 투수 쏠림 정도

- c = None이면 model_03 기준점(`fc.prior_table` 구종×손 반감기 가중 평균)과 같아야 함
- 구종×손 그룹마다 2018 초구에서 1등 투수 비중, 유효 투수 수 (= (Σn)² / Σn², 모두 같으면 투수 수)

In [7]:
overall = fp.loc[fp["GYEAR"] < fc.VAL_YEAR, L_COLS].mean().to_numpy()
S, N = fc.lookup(fc.prior_table(fp, KR, L_COLS, LOC_REF_HL), q_val, KR, L_COLS)
P03 = np.where(N > 0, S / np.maximum(N, 1e-12), overall)  # model_03 기준점
P04 = loc_league(q_val, fc.VAL_YEAR, "decay", LOC_REF_HL, None)  # c = None
print("c=None vs model_03 기준점 최대 차이", np.abs(P03 - P04).max())

d = fp[fp["GYEAR"] == 2018]
n = d.groupby(["pitch_type", "same_side", "pitcher_code"], observed=True).size()  # 그룹×투수 공 수
share = n.groupby(level=[0, 1], observed=True).apply(lambda s: pd.Series({"공": s.sum(), "투수 수": (s > 0).sum(), "1등 투수 비중": s.max() / s.sum(),
                                                                         "유효 투수 수": s.sum() ** 2 / (s ** 2).sum()})).unstack()
display(share.round(2))

c=None vs model_03 기준점 최대 차이 2.0463630789890885e-12


공  투수 수  1등 투수 비중  유효 투수 수
pitch_type same_side                                 
CHUP       0          1177.0  22.0      0.13    12.45
           1           492.0  16.0      0.21     6.92
CURV       0           486.0  28.0      0.14    11.89
           1           779.0  35.0      0.10    15.50
CUTT       0            11.0   2.0      0.64     1.86
           1            35.0   2.0      0.77     1.54
FAST       0          2991.0  38.0      0.11    23.06
           1          3260.0  38.0      0.06    28.27
FORK       0           330.0  20.0      0.14    11.65
           1           222.0  15.0      0.19     8.97
KNUC       0             3.0   1.0      1.00     1.00
SINK       0             7.0   1.0      1.00     1.00
SLID       0           580.0  36.0      0.12    16.80
           1          1429.0  36.0      0.08    24.27
TWOS       0           277.0   9.0      0.60     2.47
           1           316.0   7.0      0.60     2.41

### 🔍 대조군 재현됨, 투수 쏠림은 큰 그룹에선 작음
- c = None이면 model_03 기준점과 같음 (최대 차이 2e-12)
- 공 많은 그룹은 쏠림 작음: 직구·슬라이더·체인지업·커브 1등 투수 비중 0.06–0.21, 유효 투수 수 7–28명
- 쏠림 큰 그룹은 투수가 1–9명뿐: 투심(1등 0.60), 커터(0.64–0.77), 너클·싱커(1명)
  - 이런 그룹은 그 구종을 던지는 투수가 거의 그 투수라 동일 가중으로 바꿔도 대상이 같음

### 3-2. 기준점 무게·위치 파라미터 함께 고르기 (2019)

- 기준점이 바뀌면 투수×구종 쪽으로 당기는 강도(k_type)의 최적값도 바뀜 -> 함께 고름
- 거친 격자: 기준점 반감기 4 × 투수 무게 c 5 (0 동일 / 1 / 10 / 100 / None 공 수) × 반감기 3 × k_type 4 × k_hand 3
- 세밀한 격자: 기준점 반감기·c·반감기·k_hand (k_type은 1000 이상에서 평평해 거친 1등 고정), c = None(공 수 가중, 대조군)은 항상 후보에 포함
- 기준: 2019 24칸 log loss (실제 구종 기준), H/S/C log loss·좌표 NLL은 참고

In [8]:
def grid_search(score, grid, by="logloss"):  # grid의 모든 조합을 score로 채점 (지표 낮은 순)
    names_ = list(grid)
    combos = pd.MultiIndex.from_product(grid.values(), names=names_).to_frame(index=False)  # 후보의 모든 조합 (한 행 = 한 조합)
    rows = [{**r, **score(**r)} for r in combos.astype(object).where(combos.notna(), None).to_dict("records")]  # NaN → None으로 되돌려 넘김
    return pd.DataFrame(rows).sort_values(by).reset_index(drop=True)

def finer(cands, best, n=3):  
    # 거친 후보 목록에서 best와 양옆 후보 사이를 각각 로그 간격으로 n개씩 나눈 세밀한 후보 (best 가운데, 2n−1개)
    # best가 범위 끝이면 바깥으로 2배 넓힘, best가 None(누적/손 구분 없음)이면 None 그대로"""
    if best is None:
        return [None]
    nums = sorted(c for c in cands if c is not None)  # None 뺀 숫자 후보
    i = nums.index(best)
    lo = nums[i - 1] if i > 0 else best / 2  # 왼쪽 이웃 (없으면 절반)
    hi = nums[i + 1] if i < len(nums) - 1 else best * 2  # 오른쪽 이웃 (없으면 2배)
    vals = np.r_[np.geomspace(lo, best, n), np.geomspace(best, hi, n)[1:]]  # lo~best, best~hi를 같은 비율 간격으로
    return sorted({float(f"{v:.3g}") for v in vals})  # 유효숫자 3자리로 반올림, 중복 제거

def best_of(table, grid):  # 표 1등 행의 파라미터 dict (NaN → None, 숫자는 float)
    r = table.iloc[0]
    return {k: (None if pd.isna(r[k]) else float(r[k])) for k in grid}

def marginal(table, grid, by="logloss"):  # 파라미터마다 값별 가장 좋은 by 지표 (나머지는 그 값에서 가장 좋은 조합) -> 어느 파라미터가 민감한지
    out = {}
    for k in grid:
        s = table.assign(**{k: table[k].astype(object).where(table[k].notna(), "None")}).groupby(k, sort=False)[by].min()
        out[k] = s.reindex([("None" if v is None else v) for v in grid[k] if ("None" if v is None else v) in s.index])
    return out

In [9]:
def score_loc(half_life, k_type, k_hand, ref_hl, ref_c):  # 위치 파라미터 한 조합의 2019 채점: 24칸 log loss·top1·H/S/C + 좌표 NLL
    g = fc.gauss(loc_online(q_val, fc.VAL_YEAR, half_life, k_type, k_hand, LOC_REF, ref_hl, ref_c), q_val)  # 실제 구종 기준 정규분포
    z = fc.zone_metrics(zone_probs(g, q_val), y_zone[val])
    c = fc.coord_metrics(g, q_val)
    return {"logloss": z["logloss"], "top1": z["top1"], "H/S/C logloss": z["H/S/C logloss"],
            "H/S/C 정확도": z["H/S/C 정확도"], "좌표 NLL": c["NLL"], "평균거리 cm": c["평균거리 cm"]}

GRID_LOC = {"ref_hl": [30, 100, 300, 1000],  # 기준점 반감기 (구종×손 그룹의 공 수)
            "ref_c": [0, 1, 10, 100, None],  # 투수 무게 c (None = 공 수 가중, model_03)
            "half_life": [25, 50, 100],
            "k_type": [100, 300, 1000, 3000],
            "k_hand": [10, 30, 100]}

res = grid_search(score_loc, GRID_LOC)  # 거친 격자
best = best_of(res, GRID_LOC)
grid_f = {k: (finer(GRID_LOC[k], best[k]) if k != "k_type" else [best[k]]) for k in GRID_LOC}  # 1등 주변 세밀한 후보
if best["ref_c"] == 0:
    grid_f["ref_c"] = [0, 0.3, 1]  # 0은 로그 간격이 안 되므로 직접
grid_f["ref_c"] = sorted({c for c in grid_f["ref_c"] if c is not None}) + [None]  # 대조군(공 수 가중) 항상 포함
res_f = grid_search(score_loc, grid_f)  # 세밀한 격자
print("거친 1등", best, "→ 세밀한 후보", grid_f)
display(res_f.head(5))
for k, s in marginal(res, GRID_LOC).items():
    print("거친 ·", k, s.round(4).to_dict())  # 파라미터별 민감도 (거친 격자)
for k, s in marginal(res_f, grid_f).items():
    print("세밀 ·", k, s.round(4).to_dict())

# c(투수 무게)마다 나머지를 가장 좋게 고른 점수 -> 투수 동일 가중 쪽이 좋은지
res_all = pd.concat([res, res_f]).sort_values("logloss", ignore_index=True)  # 거친 + 세밀한 격자
by_c = res_all.assign(ref_c=res_all["ref_c"].astype(object).where(res_all["ref_c"].notna(), "None (공 수, model_03)")).groupby("ref_c", sort=False).first()
display(by_c[["ref_hl", "half_life", "k_type", "k_hand", "logloss", "H/S/C logloss", "좌표 NLL"]].assign(
    **{"model_03 대비": lambda t: t["logloss"] - score_loc(35.4, 1000.0, 17.3, 173.0, None)["logloss"]}).round(4))

거친 1등 {'ref_hl': 100.0, 'ref_c': 100.0, 'half_life': 25.0, 'k_type': 3000.0, 'k_hand': 10.0} → 세밀한 후보 {'ref_hl': [30.0, 54.8, 100.0, 173.0, 300.0], 'ref_c': [10.0, 31.6, 100.0, 141.0, 200.0, None], 'half_life': [12.5, 17.7, 25.0, 35.4, 50.0], 'k_type': [3000.0], 'k_hand': [5.0, 7.07, 10.0, 17.3, 30.0]}


,ref_hl,ref_c,half_life,k_type,k_hand,logloss,top1,H/S/C logloss,H/S/C 정확도,좌표 NLL,평균거리 cm
0,173.0,100.0,35.4,3000.0,17.3,2.893220,0.145638,1.072548,0.405038,9.252258,31.815277
1,173.0,141.0,35.4,3000.0,17.3,2.893222,0.145713,1.072539,0.405335,9.252247,31.815391
2,100.0,31.6,35.4,3000.0,17.3,2.893223,0.146084,1.072546,0.404518,9.252293,31.814772
3,173.0,200.0,35.4,3000.0,17.3,2.893228,0.145713,1.072532,0.405261,9.252242,31.815522
4,100.0,100.0,35.4,3000.0,17.3,2.893232,0.146010,1.072505,0.404889,9.252269,31.815763


거친 · ref_hl {30: 2.8956, 100: 2.8943, 300: 2.8943, 1000: 2.8951}
거친 · ref_c {0: 2.8962, 1: 2.895, 10: 2.8943, 100: 2.8943, 'None': 2.8943}
거친 · half_life {25: 2.8943, 50: 2.8945, 100: 2.8969}
거친 · k_type {100: 2.8951, 300: 2.8945, 1000: 2.8943, 3000: 2.8943}
거친 · k_hand {10: 2.8943, 30: 2.8945, 100: 2.9007}
세밀 · ref_hl {30.0: 2.8949, 54.8: 2.8937, 100.0: 2.8932, 173.0: 2.8932, 300.0: 2.8935}
세밀 · ref_c {10.0: 2.8933, 31.6: 2.8932, 100.0: 2.8932, 141.0: 2.8932, 200.0: 2.8932, 'None': 2.8933}
세밀 · half_life {12.5: 2.898, 17.7: 2.8951, 25.0: 2.8935, 35.4: 2.8932, 50.0: 2.894}
세밀 · k_type {3000.0: 2.8932}
세밀 · k_hand {5.0: 2.8981, 7.07: 2.8959, 10.0: 2.8942, 17.3: 2.8932, 30.0: 2.8947}


,ref_hl,half_life,k_type,k_hand,logloss,H/S/C logloss,좌표 NLL,model_03 대비
ref_c,,,,,,,,
100.0,173.0,35.4,3000.0,17.3,2.8932,1.0725,9.2523,-0.0000
141.0,173.0,35.4,3000.0,17.3,2.8932,1.0725,9.2522,-0.0000
31.6,100.0,35.4,3000.0,17.3,2.8932,1.0725,9.2523,-0.0000
200.0,173.0,35.4,3000.0,17.3,2.8932,1.0725,9.2522,-0.0000
"None (공 수, model_03)",173.0,35.4,3000.0,17.3,2.8933,1.0725,9.2523,0.0000
10.0,100.0,35.4,3000.0,17.3,2.8933,1.0727,9.2525,0.0001
1.0,100.0,25.0,3000.0,10.0,2.8950,1.0733,9.2560,0.0017
0.0,300.0,25.0,300.0,10.0,2.8962,1.0737,9.2580,0.0029


### 🥇1등🥇
#### 기준점 반감기 173, c 100, 반감 35.4, k_type 3000, k_hand 17.3
#### -> 2019 24칸 log loss 2.89322 (model_03 2.89327, −0.00006)
- 1등 c = 100 → 투수 무게 n/(n+100): 한 그룹 안 가중 공 수가 대부분 100 아래라 거의 공 수 비례 = model_03과 사실상 같음
- c 10–200, None 모두 2.8932–2.8933으로 평평, k_type 1000 이상도 평평 (3000 = model_03 1000과 같은 점수)
- 투수 동일 가중 쪽은 나빠짐: c = 0 +0.0029, c = 1 +0.0017 (나머지 파라미터를 다시 골라도)
  - 공 적은 투수의 흔들리는 평균에 같은 표를 주면 기준점이 흔들림
- 처음 실행 때는 거친 격자에 반감 35.4가 없고 세밀 격자에서 반감을 고정해 1등이 model_03보다 나빴음 → 세밀 격자에 반감 포함, c = None 항상 후보로 고쳐 다시 실행

In [10]:
# 선택: 2019 24칸 log loss 1등 (실전 예측은 이 값 사용)
b = best_of(res_f, grid_f)
LOC_REF_HL, REF_C = b["ref_hl"], b["ref_c"]
LOC_HL, K_TYPE, K_HAND = b["half_life"], b["k_type"], b["k_hand"]
print(f"선택: 기준점 반감기 {LOC_REF_HL}, 투수 무게 c={REF_C}, half_life={LOC_HL}, k_type={K_TYPE}, k_hand={K_HAND}")

선택: 기준점 반감기 173.0, 투수 무게 c=100.0, half_life=35.4, k_type=3000.0, k_hand=17.3


## 4. 실전 예측 (예측 구종으로 위치 이어서)

- model_03과 같음, 기준점에 투수 무게(`REF_C`)만 추가

In [11]:
def predict(q, year):  # 초구 q의 공별 예측·점수
    n, i = len(q), np.arange(len(q))
    yt, yz = q["pitch_type"].map(fc.TYPE_IDX).to_numpy(), q["zone"].map(fc.ZONE_IDX).to_numpy()  # 실제 구종·칸 번호
    nll = lambda p: -np.log(np.clip(p, 1e-12, None))  # 공별 log loss (prob_metrics와 같은 방식으로 자름)

    # ① 구종 확률 (n, 9)
    Pt = type_online(q, year, TYPE_HL, TYPE_ALPHA, TYPE_HAND_K)

    # ② 위치: 구종 9개 각각 가정 -> 24칸 확률, 실제 위치의 로그 밀도, 예측 중심 (좌우, 높이)
    PZ, LD, MX, MZ = [], [], [], []
    for t in CLASSES:
        P, g = loc_given_type(q, year, t, LOC_HL, K_TYPE, K_HAND, LOC_REF, LOC_REF_HL, ref_c=REF_C)
        PZ.append(P); LD.append(fc.log_dens(g, q)); MX.append(g[0]); MZ.append(g[1])
    PZ, LD, MX, MZ = np.stack(PZ, 1), np.stack(LD, 1), np.stack(MX, 1), np.stack(MZ, 1)  # (n, 구종, ...)
    joint = Pt[:, :, None] * PZ  # 구종 확률 × 그 구종일 때 칸 확률 = 구종×칸 결합 확률 (n, 9, 24)
    P_zone = joint.sum(1)  # 실전 24칸 확률 = 구종으로 합침
    # 좌표: 구종 확률로 섞은 정규분포에서 실제 위치의 로그 밀도
    ld_mix = np.log(np.einsum("nt,nt->n", Pt, np.exp(LD - LD.max(1, keepdims=True)))) + LD.max(1)  #  공마다 최댓값 빼기 -> exp -> 섞기 -> log (계산 안전)
    dist_true = np.hypot(q["x"] - MX[i, yt], q["CROSS_PLATE_Z_CM"] - MZ[i, yt])  # 실제 구종 중심과의 거리 cm
    dist_mix = np.hypot(q["x"] - (Pt * MX).sum(1), q["CROSS_PLATE_Z_CM"] - (Pt * MZ).sum(1))  # 구종 확률로 가중한 중심과의 거리 cm

    t_hat = np.array(CLASSES)[Pt.argmax(1)]  # 1등 구종 이름

    # 공별 결과 표 (필수: 키, 구종 확률 9개, 실전 24칸 확률 / 나머지는 선택 열)
    return pd.concat([
        q[["gameID", "pit_no", "pitcher_code", "GYEAR", "pitch_type", "zone"]].reset_index(drop=True),
        pd.DataFrame(Pt, columns=fc.P_TYPE),  # p_type_<구종>
        pd.DataFrame(P_zone, columns=fc.P_ZONE),  # p_zone_<칸>
        pd.DataFrame({"t_hat": t_hat}),  # 1등 구종 이름
        pd.DataFrame({
            "ll_type": nll(Pt[i, yt]),  # 구종 log loss
            "ll_zone": nll(P_zone[i, yz]),  # 24칸 log loss (실전)
            "ll_zone_true": nll(PZ[i, yt, yz]),  # 24칸 log loss (실제 구종)
            "ll_joint": nll(joint[i, yt, yz]),  # 구종×칸 log loss
            "nll_coord": -ld_mix,  # 좌표 NLL (실전)
            "nll_coord_true": -LD[i, yt],  # 좌표 NLL (실제 구종)
            "dist": np.asarray(dist_mix),  # 좌표 거리 cm (실전)
            "dist_true": np.asarray(dist_true),  # 좌표 거리 cm (실제 구종)
        })], axis=1)

In [12]:
# 2019 검증 + 2020 평가 초구 전부 예측
pp = pd.concat([predict(q_val, fc.VAL_YEAR), predict(q_test, fc.TEST_YEAR)], ignore_index=True)
print(pp.shape)

(26073, 48)


## 5. 평가

- 채점은 `fc.evaluate`, model_03과 나란히
- 2020은 확인용 → 2020을 보고 파라미터 다시 고르지 않음

In [13]:
def eval_years(pp_):  # 2019·2020 주요 지표 표
    return pd.DataFrame({"검증 2019": fc.evaluate(pp_, fp, fc.VAL_YEAR), "평가 2020": fc.evaluate(pp_, fp, fc.TEST_YEAR)})

summary = eval_years(pp)
m03_sum = eval_years(pd.read_parquet("results/model_03/per_pitch.parquet"))  # model_03 공별 예측
display(pd.concat({MODEL: summary, "model_03": m03_sum, "차이 (model_03 대비)": summary - m03_sum}, axis=1).round(4))

model_04          model_03          차이 (model_03 대비)        
                     검증 2019  평가 2020  검증 2019  평가 2020          검증 2019 평가 2020
구종 logloss            1.0982   1.0956   1.0982   1.0956           0.0000  0.0000
구종 top1               0.5258   0.5233   0.5258   0.5233           0.0000  0.0000
구종 top2               0.8157   0.8162   0.8157   0.8162           0.0000  0.0000
구종 top3               0.9331   0.9359   0.9331   0.9359           0.0000  0.0000
구종 macro F1           0.2476   0.2857   0.2476   0.2857           0.0000  0.0000
24칸 logloss (실전)      2.9514   2.9479   2.9512   2.9479           0.0002  0.0000
24칸 top1              0.1283   0.1333   0.1286   0.1330          -0.0004  0.0002
24칸 top3              0.2870   0.2901   0.2873   0.2897          -0.0003  0.0005
H/S/C logloss         1.0753   1.0771   1.0753   1.0772           0.0001 -0.0000
H/S/C 정확도             0.3931   0.3877   0.3935   0.3885          -0.0004 -0.0008
24칸 logloss (실제 구종)   2.8932   2.8967   2.8933   2.8967          -0.0000  0.0000
구종×칸 logloss          3.9914   3.9916   3.9914   3.9916          -0.0000  0.0000
좌표 평균거리 cm           32.7403  31.8827  32.7396  31.8837           0.0007 -0.0010
좌표 NLL                9.3197   9.2689   9.3195   9.2690           0.0002 -0.0000

- 2020 24칸 log loss 실전 2.9479 → 2.9479 (0.0000), 실제 구종 2.8967 → 2.8967 (0.0000)
- 2019 실전은 +0.0002, H/S/C 정확도 2019 −0.0004 / 2020 −0.0008
- 경기 단위 부트스트랩 (model_04 − model_03, 2020): 모든 점수 차이 불확실 (실전 24칸 0.0000, 95% −0.0001 ~ +0.0001)
- -> 투수 동일 가중은 효과 없음, 1등(c = 100)은 model_03과 사실상 같은 모델
- **채택 안 함 → model_03 유지**: 2019 차이 −0.00006은 격자 잡음 수준, 파라미터만 하나 늘어남

## 6. 저장

- `results/model_04/` → `compare.ipynb` 실행하면 자동 포함 (부트스트랩 판정은 compare 2-1에서)

In [ ]:
fc.save_results(MODEL, pp, {"summary": summary}) 

저장: results\model_04 ['per_pitch.parquet', 'summary.csv']
